In [1]:
%reload_ext autoreload
from urllib.request import thishost
%autoreload 2
import os, sys, random
import numpy as np
import pandas as pd
import seaborn as sns

from scipy.stats import zscore
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import gridspec, rcParams
from datetime import datetime as dt, timedelta
import pingouin as pg
import scipy.stats as stats
from fish import Gafftopsail
sys.path.append(r'C://Users//Zichen//anaconda3//envs//2ptank//Lib//site-packages//')
import utils
path = ('C://Data//Imaging//250821_overlap//fish4//')
rcParams['font.size'] = 14  

In [2]:
fish = Gafftopsail(path, filelist = ['stimulus', 'tail', 'eye', 'imaging', 'processed_tail'])

In [5]:
#officially start plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from PIL import Image
import io, base64

__PLOT ALL DATA__

In [8]:
def color_stims_plotly(fig, stim_name, starttime, duration, stationary_time):
    if type(stim_name) == list:
        for babystim, stationary_time in zip(stim_name, stationary_time):
            fig = color_stims_plotly(fig, babystim, starttime, duration, stationary_time)
    else:
        if 'dot' in stim_name:
            color = utils.stim_colors[stim_name.split(',')[0]]
            rown = 1
        else:
            color = utils.stim_colors[stim_name]
            rown = 2
        fig.add_shape(y0 = 0, y1 = 1,
                      x0 = starttime + stationary_time, x1 = starttime + duration,
                      fillcolor = f'rgb{color}', line = {'width': 0}, opacity = 0.5,
                      row = rown, col = 1)
    return fig

def overview_plot(fish):
    fig = make_subplots(rows = 4, cols = 2, shared_xaxes = True, row_heights = [1, 1, 2, 5],
                        specs = [[{'rowspan': 1, 'colspan': 1}, {'rowspan': 4, 'colspan': 1, 'type': 'scatter3d'}], [{'rowspan': 1, 'colspan': 1}, None], [{'rowspan': 1, 'colspan': 1}, None], [{'rowspan': 1, 'colspan': 1}, None]])
    #plot stimulus in the first two rows
    for n, row in fish.stimulus_df.iterrows():
        stim_name = row['stim_name']
        stim_starttime = row['real_starttime_s']
        duration = row['duration']
        stationary_time = row['stationary_time']
        fig = color_stims_plotly(fig, stim_name, stim_starttime, duration, stationary_time)
    fig.update_xaxes(row = 1, col =1, range = (min(fish.frametimes), max(fish.frametimes)))
    fig.update_xaxes(row = 2, col =1, range = (min(fish.frametimes), max(fish.frametimes)))
    fig.update_yaxes(row = 1, col =1, range = (0, 1), fixedrange = True)
    fig.update_yaxes(row = 2, col =1, range = (0, 1), fixedrange = True)
    
    #plot tail
    fig.add_trace(go.Heatmap(y = list(range(5)), x = fish.tail_df['real_time_s'], z = fish.tail_df[[col for col in fish.tail_df.columns if 'theta' in col]].T, showscale = False, colorscale = 'balance'), row = 3, col =1)
    fig.update_yaxes(row = 3, col =1, fixedrange = True, showticklabels = False)
    
    #plot neuron
    middle_plane = fish.planes[len(fish.planes)//2]
    f_all = pd.concat([fish.f_dict[plane] for plane in fish.planes], axis = 0)
    fig.add_trace(go.Heatmap(y=list(range(len(f_all))), x = fish.frametimes_dict[middle_plane], showscale = False,
                             z = f_all, colorscale='gray', zmin = 0, zmax = 1, ), row=4, col=1)
    fig.update_yaxes(row=4, col=1, fixedrange=True, showticklabels = False)
    
    #plot scater plot
    pos_all = pd.concat([fish.pos_dict[plane] for plane in fish.planes], axis = 0)
    fig.add_trace(go.Scatter3d(x = pos_all['xpos'], y = pos_all['ypos'], z = pos_all['zpos'], 
                               mode = 'markers', marker = {'size': 2, 'color': "yellow", 'opacity': 0.5}), row = 1, col = 2)
    fig.add_trace(go.Surface(z = np.full_like(fish.img_dict[middle_plane], 0), surfacecolor = fish.img_dict[middle_plane], 
                             showscale = False, colorscale = 'gray'), row = 1, col = 2)
    fig.update_layout({ "xaxis": {"matches": "x"},"xaxis2": {"matches": "x"}, "xaxis3":  {"matches": "x"}, "xaxis4":  {"matches": "x"}})
    fig.update_scenes(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)
    fig.update_layout(scene_aspectmode='manual', scene_aspectratio=dict(x=1, y=1.8, z=0.6))
    
    fig.write_html(fish.path + "Graphs//00_overview.html")

In [9]:
overview_plot(fish)